# 논리집 EDA 시작 노트북

저장소의 논리집 데이터를 표로 펼쳐(`python -m eda.export` → `out/eda/*.csv`) 처음 둘러보는 노트북.
각 절은 출발점일 뿐이다 — 셀을 복사해 원하는 쪽으로 파고들면 된다. 표 설명은 `eda/CLAUDE.md`.

- 1절 논리집(knowledge): 분야 × 노드 종류, 연결, 판정
- 2절 작업 기록(agent_usage): 예측 구간에 실측이 들었나, 어떤 규칙이 자주 틀렸나
- 3절 피드백 백서: 출처·자리·색인 낱말

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# 저장소 루트 찾기 (노트북을 eda/ 에서 열든 루트에서 열든)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "proviso").is_dir() and (p / "domains").is_dir())
sys.path.insert(0, str(ROOT))

# 한글 글꼴: Windows 맑은 고딕 → macOS → 리눅스 나눔 순으로 있는 것을 쓴다
plt.rcParams["font.family"] = ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

In [ ]:
# 표를 다시 만든다 (저장소 파일이 바뀌었으면 늘 최신으로). 몇 초 걸린다
from eda import export
counts = export.export(ROOT / "out" / "eda")
counts

In [ ]:
OUT = ROOT / "out" / "eda"
T = {p.stem: pd.read_csv(p, low_memory=False) for p in sorted(OUT.glob("*.csv"))}
nodes, edges, judgments = T["nodes"], T["edges"], T["judgments"]
disp, rules, fb = T["dispatches"], T["dispatch_rules"], T["feedback"]
{k: v.shape for k, v in T.items()}

## 1. 논리집 (knowledge)

In [ ]:
# 분야 × 노드 종류. far_fields(먼 분야 조각, 가설 1만여 개)가 크기를 압도하니 따로 보는 게 낫다
pd.crosstab(nodes["domain"], nodes["kind"], margins=True)

In [ ]:
# far_fields를 뺀 '내 분야'만 따로 쓰기 쉽게
mine = nodes[nodes["domain"] != "far_fields"]
claims = mine[mine["kind"].isin(["rule", "hypothesis"])]
# 살아 있는 가설 vs 이전 버전(retired), 분야별
pd.crosstab(claims["domain"], claims["alive"].map({True: "지금", False: "이전 버전"}))

In [ ]:
# 시험 수단 없는 살아 있는 가설 (자동 검사도 판정도 없음 = audit 경고 대상)
untested = claims[(claims["alive"] == True) & (claims["n_checks"].fillna(0) == 0) & (claims["n_judgments"].fillna(0) == 0)]
untested.groupby("domain").size().rename("시험 수단 없음")

In [ ]:
# 가장 많이 가리켜진 노드 (in_degree) — 조건·근거로 여러 규칙에 걸리는 것. 분야별 상위 5
(mine.sort_values("in_degree", ascending=False)
     .groupby("domain").head(5)[["domain", "kind", "id", "in_degree", "text"]])

In [ ]:
# 연결 종류(칸 이름)별 개수, 분야별
pd.crosstab(edges["field"], edges["domain"])

In [ ]:
# 판정: 출처 × 판정, 분야별 (출처 무게: user > replay > measured > auto > simulation > ai)
judgments.groupby(["domain", "source", "verdict"]).size().unstack(fill_value=0)

In [ ]:
# far_fields: 묶음(group)별 조각 수와 확신도
ff = nodes[(nodes["domain"] == "far_fields") & (nodes["kind"] == "hypothesis")]
pd.crosstab(ff["group"], ff["confidence"], margins=True).sort_values("All", ascending=False).head(25)

## 2. 작업 기록 (agent_usage — 예측 → 실측)

In [ ]:
disp["date"] = pd.to_datetime(disp["date"])
done = disp[disp["actual.minutes"].notna()].copy()
print(len(disp), "건 중 실측 있음", len(done))
# 예측 구간 안에 실측이 들었나 (hit.*: True/False, 예측이나 실측이 없으면 빈칸)
done[["hit.minutes", "hit.turns", "hit.tokens_m"]].apply(lambda s: s.dropna().astype(bool).mean()).rename("적중률")

In [ ]:
# 예측 구간 vs 실측 (분, 로그 축). 세로 막대 = 예측 구간, 점 = 실측
d = done.dropna(subset=["predicted.minutes_lo", "predicted.minutes_hi"]).sort_values("date").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(12, 4))
ax.vlines(d.index, d["predicted.minutes_lo"], d["predicted.minutes_hi"], color="#9aa3b1", lw=2, label="예측 구간")
hit = d["hit.minutes"].astype("boolean")
ax.scatter(d.index[hit == True], d.loc[hit == True, "actual.minutes"], s=10, color="#2b7a4b", label="실측(안)")
ax.scatter(d.index[hit == False], d.loc[hit == False, "actual.minutes"], s=10, color="#c0432e", label="실측(밖)")
ax.set_yscale("log"); ax.set_xlabel("기록 (날짜 순)"); ax.set_ylabel("분"); ax.legend()
plt.show()

In [ ]:
# 적중률(분)이 시간에 따라 나아졌나 — 30건 이동 평균
h = d["hit.minutes"].astype(float)
ax = h.rolling(30, min_periods=10).mean().plot(figsize=(12, 3), title="분 예측 적중률 (30건 이동 평균)")
ax.set_ylim(0, 1); plt.show()

In [ ]:
# 모델·effort별 적중률과 건수
g = done.groupby(["model", "effort"], dropna=False)
pd.DataFrame({"건수": g.size(), "분 적중률": g["hit.minutes"].apply(lambda s: s.dropna().astype(bool).mean())}).sort_values("건수", ascending=False)

In [ ]:
# 빗나간 쪽: 실측 / 예측 가운데 값(기하 평균). 1보다 크면 적게 잡은 것
import numpy as np
ok = d["predicted.minutes_lo"] > 0  # 하한 0인 예측은 가운데 값을 못 낸다
d.loc[ok, "ratio"] = d.loc[ok, "actual.minutes"] / np.sqrt(d.loc[ok, "predicted.minutes_lo"] * d.loc[ok, "predicted.minutes_hi"])
ax = np.log2(d["ratio"].dropna()).plot.hist(bins=40, figsize=(8, 3), title="log2(실측 / 예측 가운데)  — 0 = 정확")
plt.show()
d["ratio"].describe()

In [ ]:
# 빗나간 까닭(review.why)
done["review.why"].value_counts()

In [ ]:
# 규칙별: 예측에 쓴 횟수 vs 틀렸다고 지목된 횟수 (review.wrong)
used = rules[rules["role"] == "predicted.rules"].groupby("rule").size().rename("쓴 횟수")
wrong = rules[rules["role"] == "review.wrong"].groupby("rule").size().rename("틀림 지목")
r = pd.concat([used, wrong], axis=1).fillna(0).astype(int)
r["틀림 비율"] = (r["틀림 지목"] / r["쓴 횟수"].where(r["쓴 횟수"] > 0)).round(2)
r.sort_values("쓴 횟수", ascending=False).head(20)

In [ ]:
# 상황 값(S_*)별 실측 분의 가운데 값 — 어떤 스위치가 시간과 같이 움직이나 (인과 아님, 선택 편향 주의 F084)
s_cols = [c for c in done.columns if c.startswith("situation.S_")]
rows = []
for c in s_cols:
    m = done.groupby(c)["actual.minutes"].agg(["count", "median"])
    if {"예", "아니오"} <= set(m.index) and m.loc[["예", "아니오"], "count"].min() >= 10:
        rows.append({"상황": c.removeprefix("situation."), "예 n": m.loc["예", "count"], "아니오 n": m.loc["아니오", "count"],
                     "예 중앙값": m.loc["예", "median"], "아니오 중앙값": m.loc["아니오", "median"],
                     "배수": round(m.loc["예", "median"] / m.loc["아니오", "median"], 2)})
pd.DataFrame(rows).sort_values("배수", ascending=False)

## 3. 피드백 백서

In [ ]:
pd.crosstab(fb["source_kind"], fb["place_kind"], margins=True)

In [ ]:
# 색인 낱말별 항목 수
T["feedback_index"]["word"].value_counts().head(20)

In [ ]:
# 날짜별로 쌓인 항목 수 (출처 글의 첫 날짜)
fb["date"] = pd.to_datetime(fb["date"], errors="coerce")
fb.set_index("date").groupby("source_kind").resample("W").size().unstack(0, fill_value=0)